# S&P500/VIX Report Figures

Report-only notebook for CS673/public figures. It does not train models by default. It reads generated figures from ignored `outputs/` paths and can regenerate diagnostics only when explicitly enabled.


## Setup


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Markdown, display

RUN_SMOKE = False
RUN_TRAINING = False
RUN_EVALUATION = False


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_yaml(path: str | Path) -> dict[str, Any]:
    resolved = repo_path(path)
    if not resolved.exists():
        return {}
    loaded = yaml.safe_load(resolved.read_text())
    return loaded if isinstance(loaded, dict) else {}


def load_json(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


def print_command(command: list[str]) -> None:
    print(" ".join(shlex.quote(part) for part in command))


def maybe_run(command: list[str], *, enabled: bool, label: str) -> None:
    print(f"{label} command:")
    print_command(command)
    if enabled:
        subprocess.run(command, cwd=REPO_ROOT, check=True)
    else:
        print(f"{label} skipped; enable the matching RUN_* flag to execute it.")


print(f"Repository root: {REPO_ROOT}")
from IPython.display import Image

REGENERATE_GEOMETRY = False
RUN_PAPER_EVALUATION = False
REGENERATE_PATH_PANEL = False

STANDARD_GEOMETRY_DIR = Path("outputs/latent_geometry/sp500_vix_standard_vq")
RVQ_Q2_GEOMETRY_DIR = Path("outputs/latent_geometry/sp500_vix_rvq_q2")
PAPER_STYLE_DIR = Path("outputs/sp500_vix_discrete/paper_style")
TOKENIZER_DIR = Path("outputs/sp500_vix_discrete/tokenizer/sp500_vix_causal_vq_tokenizer_seed0")
TOKEN_DATA_DIR = Path("outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16")
TOKEN_PRIOR_DIR = Path("outputs/sp500_vix_discrete/token_prior/additive/<prior-run>")
BASE_DATA_DIR = Path("data/processed")

## Figure Manifest


In [ ]:
figure_manifest = [
    (
        "paper",
        PAPER_STYLE_DIR / "real_vs_generated_paths.png",
        "Real vs generated S&P500/VIX paths, if generated by the final evaluation run.",
    ),
    ("paper", PAPER_STYLE_DIR / "returns_distribution.png", "One-step return distribution."),
    ("paper", PAPER_STYLE_DIR / "volatility_distribution.png", "Path-volatility distribution."),
    (
        "paper",
        PAPER_STYLE_DIR / "squared_return_autocorrelation.png",
        "Squared-return autocorrelation.",
    ),
    ("paper", PAPER_STYLE_DIR / "vix_bucket_paths.png", "Generated paths by VIX bucket."),
    (
        "geometry",
        STANDARD_GEOMETRY_DIR / "codebook_projection.png",
        "Standard VQ codebook projection.",
    ),
    (
        "geometry",
        STANDARD_GEOMETRY_DIR / "codebook_usage_projection.png",
        "Standard VQ usage projection.",
    ),
    ("geometry", STANDARD_GEOMETRY_DIR / "vix_bucket_code_usage.png", "VIX-bucket code usage."),
    (
        "geometry",
        STANDARD_GEOMETRY_DIR / "token_trajectory_examples.png",
        "Token trajectory examples.",
    ),
    (
        "ablation",
        RVQ_Q2_GEOMETRY_DIR / "q0_q1_pair_heatmap.png",
        "RVQ q0/q1 heatmap as ablation evidence.",
    ),
]
rows = [
    {
        "role": role,
        "path": display_path(path),
        "exists": repo_path(path).exists(),
        "caption": caption,
    }
    for role, path, caption in figure_manifest
]
display(pd.DataFrame(rows))

## Display Existing Figures


In [ ]:
for role, path, caption in figure_manifest:
    resolved = repo_path(path)
    if not resolved.exists():
        print(f"missing: {display_path(path)}")
        continue
    display(Markdown(f"### {role}: `{display_path(path)}`"))
    display(Markdown(caption))
    display(Image(filename=str(resolved)))

## Optional Real vs Generated Path Panel

This cell reads saved paper-style tensors and writes a small path panel only when `REGENERATE_PATH_PANEL=True`.


In [ ]:
if REGENERATE_PATH_PANEL:
    import matplotlib.pyplot as plt
    import torch

    batch_path = repo_path(PAPER_STYLE_DIR / "discrete_paper_style_batch.pt")
    if not batch_path.exists():
        print(f"Missing saved tensor batch: {display_path(batch_path)}")
    else:
        payload = torch.load(batch_path, map_location="cpu", weights_only=True)

        def first_tensor(keys: list[str]):
            for key in keys:
                value = payload.get(key)
                if value is not None:
                    return value
            return None

        real = first_tensor(["real_paths", "real_data"])
        generated = first_tensor(["decoded_paths", "discrete_paths", "fake_data"])
        if real is None or generated is None:
            print("Saved batch does not contain recognised real/generated path tensors.")
        else:
            real_2d = real.detach().float()
            generated_2d = generated.detach().float()
            if real_2d.ndim == 3:
                real_2d = real_2d[..., 0]
            if generated_2d.ndim == 3:
                generated_2d = generated_2d[..., 0]
            fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True, sharey=True)
            axes[0].plot(real_2d[:24].T, alpha=0.45, linewidth=0.9)
            axes[0].set_title("Real")
            axes[1].plot(generated_2d[:24].T, alpha=0.45, linewidth=0.9)
            axes[1].set_title("Generated")
            for axis in axes:
                axis.set_xlabel("Time")
            axes[0].set_ylabel("Normalised level")
            fig.tight_layout()
            output_path = repo_path(PAPER_STYLE_DIR / "real_vs_generated_paths.png")
            output_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_path)
            plt.close(fig)
            print(f"Wrote {display_path(output_path)}")
else:
    print(
        "Path panel regeneration skipped; set REGENERATE_PATH_PANEL=True to create it from saved tensors."
    )

## Optional Regeneration Commands

These commands are printed every time and run only when the corresponding flags are enabled. Replace prior/checkpoint placeholders before running paper-style evaluation.


In [ ]:
geometry_command = [
    "poetry",
    "run",
    "python",
    "scripts/analyze_discrete_latent_geometry.py",
    "--config",
    "configs/experiments/sp500_vix_causal_vq_tokenizer.yaml",
    "--tokenizer-dir",
    display_path(TOKENIZER_DIR),
    "--token-data-dir",
    display_path(TOKEN_DATA_DIR),
    "--output-dir",
    display_path(STANDARD_GEOMETRY_DIR),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--plot-voronoi",
]
paper_command = [
    "poetry",
    "run",
    "python",
    "scripts/evaluate_sp500_vix_paper_style.py",
    "--discrete-config",
    "configs/experiments/sp500_vix_causal_token_prior_additive.yaml",
    "--discrete-prior-dir",
    display_path(TOKEN_PRIOR_DIR),
    "--discrete-tokenizer-dir",
    display_path(TOKENIZER_DIR),
    "--continuous-config",
    "configs/experiments/sp500_vix_beta_cvae.yaml",
    "--continuous-model-dir",
    "<continuous-final-model-dir>",
    "--output-dir",
    display_path(PAPER_STYLE_DIR),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--n-sample",
    "1000",
    "--seed",
    "99",
    "--temperature",
    "1.0",
    "--top-k",
    "40",
]
maybe_run(geometry_command, enabled=REGENERATE_GEOMETRY, label="Standard VQ latent geometry")
maybe_run(
    paper_command,
    enabled=RUN_PAPER_EVALUATION and "<" not in str(TOKEN_PRIOR_DIR),
    label="Paper-style evaluation",
)

## Interpretation

Standard VQ is the promoted public method because it gives broad code utilisation, VIX-sensitive code usage, and the simplest one-code-per-time-step interface for the additive scalar-conditioned causal AR prior.

RVQ q2 is an ablation. Its q0/q1 heatmap is useful evidence for the sparse same-time joint-support problem that makes multi-code generation harder, but it is not the final architecture.
